# 👩🏻👨🏻👧🏻👦🏻👶🏻 <span style="color: white; background-color: Crimson"><b> Extração do Relatório DEPENDENTES </b></span></p>

🧩 <span style="color: Salmon"><b> 1- Configuração inicial e registro de processo </b></span></p>
- O script inicia com importações de bibliotecas (Selenium, PyAutoGUI, Pandas, OpenPyXL, logging etc.)  
- Definição do dicionário CONFIG (paths de origem, destino, arquivos movidos e processados)  
- Registro da Etapa 0 no arquivo PROCESSOS.xlsx
- Isso estabelece rastreabilidade, requisito essencial para auditoria e governança dos processos de RH

🌐 <span style="color: Salmon"><b> 2- Acesso automático ao Datamace </b></span></p>
- A automação verifica se a janela HTML5 já está aberta
- Ativa a janela  
- Executa login interno via PyAutoGUI
- Se não estiver aberta, abre o navegador via Selenium  
- Acessa o portal Cloud do Datamace  
- Faz login inicial  
- Aguarda a abertura da interface HTML5  
- Realiza o segundo login automático
- Esse fluxo permite acesso estável e reprodutível ao Datamace

🧭 <span style="color: Salmon"><b> 3- Navegação até o relatório de Dependentes </b></span></p>
- Após o login, o script navega automaticamente por CS → Cadastros → Dependentes → Relatórios → Dependentes por Colaborador (Excel)
- Define filtros necessários  
- Seleciona exportação em Excel  
- Confirma a geração do arquivo
- O relatório é baixado diretamente para a pasta de Downloads

📥 <span style="color: Salmon"><b> 4- Confirmação e validação do download </b></span></p>
- O pipeline exibe uma janela PyAutoGUI “O download da base DEPENDENTES foi realizado?”
- A automação só continua após essa confirmação, garantindo que o arquivo realmente está disponível

🧹 <span style="color: Salmon"><b> 5- Tratamento completo da base de Dependentes </b></span></p>
- Ao identificar arquivos iniciados com o padrão REL-DEPENDENTES
- o script realiza mapeamento das colunas
- Converte campos brutos para nomes padronizados
- Conversão de datas
- Converte colunas para datetime com tratamento de erros
- Padronização de valores
- Limpa:
    - Espaços extras  
    - Caracteres inválidos  
    - Strings inconsistentes
- Normalização da estrutura
- Garante que todos os arquivos seguem um padrão único antes da consolidação

📦 <span style="color: Salmon"><b> 6- Consolidação de múltiplos arquivos </b></span></p> 
- A automação processa cada arquivo individualmente  
- Empilha todas as bases tratadas (concat)  
- Remove duplicidades  
- Ordena colunas conforme padrão interno  
- Registra Etapa 1 no PROCESSOS.xlsx
- O resultado é uma base única, consolidada e limpa

📊 <span style="color: Salmon"><b> 7- Geração do Excel final formatado </b></span></p>
- Cria o arquivo DEPENDENTES.xlsx
- Com aba única  
- Estrutura tabular Excel  
- Formatação TableStyleLight13  
- Qualidade adequada para relatórios, auditorias e integrações internas

📁 <span style="color: Salmon"><b> 8- Arquivamento dos arquivos brutos </b></span></p>
- Após processamento todos os arquivos são movidos para ARQUIVOS MOVIDOS  
- Mantendo trilha histórica e evitando reprocessamentos acidentais

🗃️ <span style="color: Salmon"><b> 9- Exportação para base do banco de dados </b></span></p>
- O script gera tb_dependentes.csv
- Usado em:
    - Pipelines SQL
    - ETL  
    - DW  
    - Dashboards

🧾 <span style="color: Salmon"><b> 10- Finalização e registro da execução </b></span></p>
- No encerramento calcula o tempo total  
- Registra Etapa 2 no PROCESSOS.xlsx  
- Exibe mensagem consolidada de conclusão

# Importando as Bibliotecas

In [1]:
import logging
import os
import pandas as pd
import pyautogui
import pygetwindow as gw
import pyperclip
import shutil
import socket
import sys
import time
import tkinter as tk
import win32com.client
import win32gui, win32con
from asyncio.log import logger
from contextlib import contextmanager
from datetime import datetime, date
from dotenv import load_dotenv
from openpyxl import load_workbook, Workbook
from openpyxl.utils.dataframe import dataframe_to_rows
from openpyxl.worksheet.table import Table, TableStyleInfo
from pathlib import Path
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.webdriver import ActionChains
from selenium.common.exceptions import WebDriverException
from tkinter import simpledialog
from webdriver_manager.chrome import ChromeDriverManager

tempo_0 = [id, datetime.today(), 0]

# Logging (apenas console)
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s',
    handlers=[
        logging.StreamHandler()
    ]
)
logger = logging.getLogger(__name__)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Processo de Importação finalizado')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Processo de Importação finalizado

----------------------------------------------------------------------------------------------------


# Caminhos de Pastas

In [2]:
CONFIG = {
    'id_processo': 8,
    'processos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\PROCESSOS.xlsx'),
    'origem': Path(r'C:\Users\rodrigo.bernandes\Downloads'),
    'movidos': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\2. ARQUIVOS MOVIDOS'),
    'saida': Path(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\DEPENDENTES.xlsx'),
    'env_path': Path(r'X:\Gestão de Pessoas\Analytics\08 - Notebooks Python\08.3 - Automações\.env'),
    'padrao': 'GPA10B-DEP',
}

# Registra Etapa do Processo

In [3]:
def append_registro_processo(caminho, id_proc, etapa):
    try:
        wb = load_workbook(caminho)
        ws = wb['REGISTROS']
        ws.append([id_proc, datetime.today(), etapa])
        wb.save(caminho)
        wb.close()
    except Exception as e:
        print(f"❌ Erro ao registrar etapa {etapa}: {e}")

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Registro de processos')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Registro de processos

----------------------------------------------------------------------------------------------------


# Configurações Iniciais e Acessando o Datamace

In [4]:
load_dotenv(dotenv_path='env_path')

cloud_user = os.getenv("CLOUD_USER")
cloud_password = os.getenv("CLOUD_PASSWORD")
app_user = os.getenv("APP_USER")
app_password = os.getenv("APP_PASSWORD")
datamace = os.getenv("SITE_DATAMACE")

# Validar se todas existem
if not all([cloud_user, cloud_password, app_user, app_password]):
    raise ValueError("Uma ou mais variáveis de ambiente não foram encontradas. Verifique o arquivo .env.")

titulo_aba = "HTML5"

# Verifica se janela 'HTML5' está aberta
if gw.getWindowsWithTitle(titulo_aba):
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')
else:
    driver = webdriver.Chrome()
    time.sleep(1)
    window = win32gui.GetForegroundWindow()
    win32gui.ShowWindow(window, win32con.SW_MAXIMIZE)
    time.sleep(1)
    driver.get(datamace)
    time.sleep(8)
    # Login cloud
    driver.find_element(By.NAME, "username").send_keys(cloud_user)
    driver.find_element(By.NAME, "Password").send_keys(cloud_password)
    driver.find_element(By.NAME, "Password").send_keys(Keys.ENTER)
    time.sleep(30)
    # Login pyautogui
    pyautogui.click(836, 445)
    time.sleep(0.5)
    pyautogui.typewrite(app_user)
    pyautogui.press('tab')
    time.sleep(0.5)
    pyautogui.typewrite(app_password)
    pyautogui.press('enter')

print("🏁 Acesso finalizado")

🏁 Acesso finalizado


# Acessando o GIP

In [5]:
# Fechando janelas de Novidades do Datamace
time.sleep(3)
pyautogui.press('enter')
time.sleep(3)
pyautogui.click(x=1268, y=199)
# Acessando o GIP
time.sleep(10) # Tempo maior para aguardar carregar o GIP
pyautogui.click(x=168, y=185) # Maximiza a janela
time.sleep(2)
pyautogui.click(x=168, y=185) # Clica no GIP
time.sleep(5)
pyautogui.click(x=1079, y=240) # Fecha janela de Tarefas Anuais 
time.sleep(2)
pyautogui.click(x=66, y=303) # Base de Dados
time.sleep(2)
pyautogui.click(x=107, y=519) # Dependentes 
time.sleep(2)
pyautogui.click(x=329, y=545) # Rel.Dependentes
time.sleep(2)
pyautogui.click(x=571, y=290) # Multi processamento
time.sleep(2)
pyautogui.click(x=763, y=354) # Opção de saída
time.sleep(2)
pyautogui.click(x=780, y=382) # Excel
time.sleep(2)
pyautogui.click(x=790, y=374) # Situação do trabalhador
time.sleep(2)
pyautogui.click(x=760, y=419) # Todos
time.sleep(2)
pyautogui.click(x=596, y=717) # Confirmar
time.sleep(2)
pyautogui.click(x=569, y=705) # Confirmar
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Extraindo relatório')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Extraindo relatório

----------------------------------------------------------------------------------------------------


# Aguardando a Conclusão do Download da Base DEPENDENTES

In [6]:
# Configuração de logging
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(levelname)s - %(message)s")

def verificar_download_base() -> bool:
    """
    Exibe uma caixa de diálogo para confirmar se o download da base COLAB foi realizado.
    """
    tipo_escolhido = pyautogui.confirm(
        text='Foi realizado o download da base DEPENDENTES?', 
        title='Seleção de Extração', 
        buttons=['Sim']
    )

    # Se fechar a janela
    if tipo_escolhido is None:
        logging.warning("Operação cancelada pelo usuário no prompt inicial. ❌ Encerrando o script.")
        sys.exit(0)
        
    # Se for 'Sim'
    logging.info(f"Opção selecionada: {tipo_escolhido}. ✅ Continuando o processo...")
    return True

# --- Execução principal ---
if __name__ == "__main__":
    # Chama a função de validação antes de seguir com o restante do código
    verificar_download_base()
    
    logging.info("Iniciando a leitura e processamento dos dados...")

2026-06-24 11:09:56,250 - INFO - Opção selecionada: Sim. ✅ Continuando o processo...
2026-06-24 11:09:56,308 - INFO - Iniciando a leitura e processamento dos dados...


# Mapeamento de Colunas

In [7]:
MAPEAMENTO_COLUNAS = {
    'COD_EMP': 'cod_empresa',
    'NUM_TRAB': 'registro',
    'NOME_TRAB': 'nome',
    'NOME_COMP': 'nome_dependente',
    'TIPO_DEP': 'tipo_dependente',
    'SEXO_DEP': 'sexo_dependente',
    'DAT_NASCTO': 'nascimento_dependente',
    'CPF_DEP': 'cpf_dependente'
}
COLUNAS_DATA = ['DAT_NASCTO']

# Processando Dependentes

In [8]:
@contextmanager

def gerenciar_workbook(caminho: Path, sheet: str):
    """Context manager para operações seguras com workbook."""
    wb = load_workbook(caminho)
    ws = wb[sheet]
    try:
        yield ws
    finally:
        wb.save(caminho)

def registrar_etapa(caminho: Path, id_proc: int, etapa: int):
    """Registra etapa do processo."""
    with gerenciar_workbook(caminho, 'REGISTROS') as ws:
        ws.append([id_proc, datetime.today(), etapa])
    logger.debug(f"✅ Etapa {etapa} registrada")

def converter_datas(df: pd.DataFrame, colunas: list) -> pd.DataFrame:
    """Converte colunas para datetime."""
    for col in colunas:
        if col in df.columns:
            df[col] = pd.to_datetime(df[col], format='%d/%m/%Y', errors='coerce')
    return df

def mapear_colunas_acidentes(df: pd.DataFrame) -> pd.DataFrame:
    """Mapeia colunas do arquivo de acidentes."""
    colunas_existentes = {k: v for k, v in MAPEAMENTO_COLUNAS.items() if k in df.columns}
    return df.rename(columns=colunas_existentes)[list(colunas_existentes.values())]

def processar_arquivo(caminho: Path) -> pd.DataFrame:
    """Processa um arquivo de acidentes."""
    try:
        logger.debug(f"Carregando: {caminho.name}")
        
        # Carregar
        df = pd.read_excel(caminho, engine='openpyxl')
        
        # Limpar
        df = df.dropna(subset=['NOME_TRAB'])
        
        # Converter tipos
        df['NUM_TRAB'] = pd.to_numeric(df['NUM_TRAB'], errors='coerce')
        df = converter_datas(df, COLUNAS_DATA)
        
        # Mapear colunas
        df = mapear_colunas_acidentes(df)
        
        logger.debug(f"✅ {len(df)} registros processados")
        return df
        
    except Exception as e:
        logger.error(f"❌ Erro ao processar {caminho.name}: {e}")
        return None
    
def criar_excel_com_tabela(df: pd.DataFrame, caminho: Path):
    """Cria Excel com tabela formatada."""
    logger.info(f"📝 Criando Excel final ({len(df)} registros)...")
    
    # Salvar com Pandas
    df.to_excel(caminho, sheet_name='DEPENDENTES', index=False, engine='openpyxl')
    
    # Aplicar formatação
    wb = load_workbook(caminho)
    ws = wb.active
    
    tabela = Table(displayName="DEPENDENTES", ref=ws.dimensions)
    estilo = TableStyleInfo(
        name="TableStyleLight13",
        showFirstColumn=False,
        showLastColumn=False,
        showRowStripes=True,
        showColumnStripes=True
    )
    tabela.tableStyleInfo = estilo
    ws.add_table(tabela)
    
    wb.save(caminho)
    logger.info(f"✅ Excel criado: {caminho}")
    
# Main
if __name__ == "__main__":
    tempo_inicio = datetime.now()
    
    logger.info("=" * 80)
    logger.info("🔄 PROCESSAMENTO DE DEPENDENTES")
    logger.info("=" * 80)

2026-06-24 11:09:56,789 - INFO - ================================================================================
2026-06-24 11:09:56,793 - INFO - 🔄 PROCESSAMENTO DE DEPENDENTES
2026-06-24 11:09:56,796 - INFO - ================================================================================


# Processamento e Consolidação do Arquivo

In [9]:
try:
    # Etapa 1: Registrar início
    registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 0)
        
    # Etapa 2: Buscar arquivos
    logger.info("\n📂 Buscando arquivos...")
    arquivos = [f for f in CONFIG['origem'].iterdir() if f.name.startswith(CONFIG['padrao'])]
        
    if not arquivos:
        logger.warning("❌ Nenhum arquivo encontrado")
        exit()
        
    logger.info(f"✅ {len(arquivos)} arquivo(s) encontrado(s)\n")
        
    registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 1)
        
    # Etapa 3: Processar arquivos
    logger.info("🔄 Processando arquivos...\n")
        
    todas_bases = []
        
    for idx, arquivo in enumerate(arquivos, 1):
        logger.info(f"[{idx}/{len(arquivos)}] {arquivo.name}")
            
        df = processar_arquivo(arquivo)
            
        if df is not None and len(df) > 0:
            todas_bases.append(df)
                
            try:
                destino = CONFIG['movidos'] / arquivo.name
                shutil.move(str(arquivo), str(destino))
                logger.info(f"✅ Movido para arquivos processados\n")
            except Exception as e:
                logger.error(f"⚠️ Erro ao mover: {e}\n")
        else:
            logger.warning(f"❌ Sem dados válidos\n")
        
    registrar_etapa(CONFIG['processos'], CONFIG['id_processo'], 2)
        
    # Etapa 4: Consolidar e salvar
    if todas_bases:
        logger.info("💾 Consolidando dados...")
        base_final = pd.concat(todas_bases, ignore_index=True)
        base_final = base_final.drop_duplicates()
        logger.info(f"✅ {len(base_final)} registros consolidados\n")
            
        criar_excel_com_tabela(base_final, CONFIG['saida'])
    else:
        logger.error("❌ Nenhum arquivo foi processado!")
        exit()
        
    # Finalizar
    tempo_duracao = datetime.now() - tempo_inicio
        
    logger.info("\n" + "=" * 80)
    logger.info("✅ PROCESSO FINALIZADO COM SUCESSO!")
    logger.info(f"   Tempo de execução: {tempo_duracao}")
    logger.info("=" * 80)
        
except Exception as e:
    logger.error(f"\n❌ ERRO CRÍTICO: {e}")
    import traceback
    logger.error(traceback.format_exc())

2026-06-24 11:09:57,433 - INFO - 
📂 Buscando arquivos...
2026-06-24 11:09:57,437 - INFO - ✅ 1 arquivo(s) encontrado(s)

2026-06-24 11:09:57,774 - INFO - 🔄 Processando arquivos...

2026-06-24 11:09:57,776 - INFO - [1/1] GPA10B-DEP.XLSX
c:\Users\rodrigo.bernandes\AppData\Local\Programs\Python\Python312\Lib\site-packages\openpyxl\styles\stylesheet.py:237: UserWarning: Workbook contains no default style, apply openpyxl's default
  warn("Workbook contains no default style, apply openpyxl's default")
2026-06-24 11:10:12,831 - INFO - ✅ Movido para arquivos processados

2026-06-24 11:10:13,233 - INFO - 💾 Consolidando dados...
2026-06-24 11:10:13,256 - INFO - ✅ 29830 registros consolidados

2026-06-24 11:10:13,257 - INFO - 📝 Criando Excel final (29830 registros)...
2026-06-24 11:10:23,126 - INFO - ✅ Excel criado: X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\DEPENDENTES.xlsx
2026-06-24 11:10:23,128 - INFO - 
2026-06-24 11:10:23,129 - INFO - ✅ PROCESSO FINALIZADO COM SUCESSO!
2026-

# Atualizando o Arquivo Excel Controle HC e Atestados

In [10]:
# Caminho do arquivo
path_excel = r"X:\Gestão de Pessoas\Analytics\10 - Relatórios\10.4 - HC e Atestados Médicos\Controle_HC e Atestados.xlsx"
os.startfile(path_excel) # Abre o arquivo
time.sleep(60)
pyautogui.press('esc')
time.sleep(15)

# Utiliza o comando "Ir para" (Ctrl + G) para navegar até a aba e célula
pyautogui.hotkey('ctrl', 'g')
time.sleep(1)

# Digita o endereço completo
pyautogui.write('DEPENDENTES!B5')
time.sleep(1)
pyautogui.press('enter')
time.sleep(3)

#Atualizando o arquivo
pyautogui.hotkey('alt', 'f5')
time.sleep(2)

print('----------------------------------------------------------------------------------------------------')
print('')
print('   ✅ Planilha atualizada com sucesso')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

   ✅ Planilha atualizada com sucesso

----------------------------------------------------------------------------------------------------


# Criando Base em CSV para o Banco de Dados

In [11]:
# Ler o XLSX
df = pd.read_excel(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\DEPENDENTES.xlsx')

# Salvar como CSV
df.to_csv(r'X:\Gestão de Pessoas\Analytics\03 - Bases\1. BASES TRATADAS\tb_dependentes.csv', index=False, encoding='utf-8')

print("✅ Arquivo convertido para CSV com sucesso!")

✅ Arquivo convertido para CSV com sucesso!


# Resumo de Finalização do Processo

In [12]:
tempo_1 = [id, datetime.today(), 8]

print('----------------------------------------------------------------------------------------------------')
print('')
print('     ✅  Processo finalizado')
print('')
print('     ⏱️   Tempo de execução:')
print('')
print(f'   {tempo_1[1] - tempo_0[1]}')
print('')
print('----------------------------------------------------------------------------------------------------')

----------------------------------------------------------------------------------------------------

     ✅  Processo finalizado

     ⏱️   Tempo de execução:

   1:19:54.501445

----------------------------------------------------------------------------------------------------
